In [2]:
# Need to run from base environment for grib, which is miniconda3/bin/python
# to run code in terminal using that environment, run this command: /home/csutter/miniconda3/bin/python /home/csutter/DRIVE-clean/weather_events/notebooks/cam_modelpred_QPE_active.py <-- update this last part w the script you want to run

import xarray as xr
import requests
import gzip
import os
import pandas as pd
import numpy as np


Attempting S3 Download: https://noaa-mrms-pds.s3.amazonaws.com/CONUS/PrecipRate_00.00/20240115/MRMS_PrecipRate_00.00_20240115-120000.grib2.gz

✅ Success! Dataset loaded.
<xarray.Dataset>
Dimensions:         (latitude: 3500, longitude: 7000)
Coordinates:
    time            datetime64[ns] ...
    step            timedelta64[ns] ...
    heightAboveSea  float64 ...
  * latitude        (latitude) float64 54.99 54.98 54.98 ... 20.03 20.02 20.01
  * longitude       (longitude) float64 230.0 230.0 230.0 ... 300.0 300.0 300.0
    valid_time      datetime64[ns] ...
Data variables:
    unknown         (latitude, longitude) float32 ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             161
    GRIB_centreDescription:  161
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             161
    history:                 2026-03-02T16:07 GRIB to CDM+CF via cfgrib-0.9.1...


In [6]:
import sys
import xarray
print(sys.executable)
print(xarray.__file__)

/home/csutter/miniconda3/bin/python
/home/csutter/miniconda3/lib/python3.9/site-packages/xarray/__init__.py


In [7]:

def get_qpe_s3(ts):
    """
    ts: pandas Timestamp (UTC)
    """
    # 1. Format the S3 URL for NOAA Open Data
    # Path: https://noaa-mrms-pds.s3.amazonaws.com/CONUS/PrecipRate_00.00/YYYYMMDD/MRMS_PrecipRate_00.00_YYYYMMDD-HHMMSS.grib2.gz
    date_str = ts.strftime('%Y%m%d')
    # MRMS files are every 2 mins. Let's force it to 00 minutes for the test.
    time_str = ts.strftime('%H%M00') 
    
    base_url = "https://noaa-mrms-pds.s3.amazonaws.com/CONUS/PrecipRate_00.00"
    file_name = f"MRMS_PrecipRate_00.00_{date_str}-{time_str}.grib2.gz"
    full_url = f"{base_url}/{date_str}/{file_name}"
    
    print(f"Attempting S3 Download: {full_url}")
    
    # 2. Download
    r = requests.get(full_url)
    if r.status_code == 200:
        temp_grib = "test_qpe.grib2"
        with open(temp_grib, "wb") as f:
            # We decompress the .gz before saving
            f.write(gzip.decompress(r.content))
        
        # 3. Open with cfgrib (which you have!)
        ds = xr.open_dataset(temp_grib, engine="cfgrib", backend_kwargs={'indexpath': ''})
        return ds
    else:
        print(f"Failed. Status Code: {r.status_code}")
        if r.status_code == 403 or r.status_code == 404:
            print("Check: Ensure the time ends in a multiple of 2 (e.g., 02, 04, 10).")
        return None



In [8]:
# EXECUTE FUNCTION ABOVE
test_ts = pd.Timestamp("2024-01-15 12:00:00")

qpe_ds = get_qpe_s3(test_ts)

if qpe_ds:
    print("\n✅ Success! Dataset loaded.")
    print(qpe_ds)

Attempting S3 Download: https://noaa-mrms-pds.s3.amazonaws.com/CONUS/PrecipRate_00.00/20240115/MRMS_PrecipRate_00.00_20240115-120000.grib2.gz

✅ Success! Dataset loaded.
<xarray.Dataset>
Dimensions:         (latitude: 3500, longitude: 7000)
Coordinates:
    time            datetime64[ns] ...
    step            timedelta64[ns] ...
    heightAboveSea  float64 ...
  * latitude        (latitude) float64 54.99 54.98 54.98 ... 20.03 20.02 20.01
  * longitude       (longitude) float64 230.0 230.0 230.0 ... 300.0 300.0 300.0
    valid_time      datetime64[ns] ...
Data variables:
    unknown         (latitude, longitude) float32 ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             161
    GRIB_centreDescription:  161
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             161
    history:                 2026-03-02T23:06 GRIB to CDM+CF via cfgrib-0.9.1...


In [5]:
# Execute try 2

# 1. SETUP PARAMETERS
time_str = "20250412_0015"  # Your input format
THRESHOLD = 0.1             # mm/hr (The "Active" cutoff)

# 2. TIME ALIGNMENT (The 2-Minute Even Rule)
# Convert string to Timestamp
raw_ts = pd.to_datetime(time_str, format="%Y%m%d_%H%M")

# MRMS files are every 2 mins (usually even). Round down to be safe.
even_min = (raw_ts.minute // 2) * 2
mrms_ts = raw_ts.replace(minute=even_min, second=0)

print(f"Original Time: {raw_ts}")
print(f"Targeting MRMS File: {mrms_ts.strftime('%Y%m%d-%H%M00')}")

# 3. GET DATA (Using your existing function)
qpe_ds = get_qpe_s3(mrms_ts)

Original Time: 2025-04-12 00:15:00
Targeting MRMS File: 20250412-001400
Attempting S3 Download: https://noaa-mrms-pds.s3.amazonaws.com/CONUS/PrecipRate_00.00/20250412/MRMS_PrecipRate_00.00_20250412-001400.grib2.gz


In [3]:
# 1. Rename 'unknown' to 'precip_rate' for clarity
ds = qpe_ds.rename({'unknown': 'precip_rate'})

# 2. Slice to a New York bounding box
# Lat: 40 to 45, Lon: 280 to 290 (roughly -80 to -70 W)
ny_slice = ds.sel(latitude=slice(45, 40), longitude=slice(280, 290))

# 3. Check the max value
max_precip = ny_slice.precip_rate.max().values
print(f"Max Precip Rate in NY slice: {max_precip} mm/hr")

# 4. Check for 'Missing' values
# MRMS uses -999 for missing/no coverage. 
# We should mask those out.

ny_clean = ny_slice.where(ny_slice.precip_rate >= 0)

print(f"Average precip where it was actually snowing/raining: {ny_clean.precip_rate.mean().values} mm/hr")

Max Precip Rate in NY slice: 129.60000610351562 mm/hr
Average precip where it was actually snowing/raining: 0.013453599065542221 mm/hr


In [4]:
ny_clean

<xarray.Dataset>
Dimensions:         (latitude: 500, longitude: 1000)
Coordinates:
    time            datetime64[ns] 2024-01-15T12:00:00
    step            timedelta64[ns] 00:00:00
    heightAboveSea  float64 0.0
  * latitude        (latitude) float64 45.0 44.99 44.98 ... 40.03 40.02 40.01
  * longitude       (longitude) float64 280.0 280.0 280.0 ... 290.0 290.0 290.0
    valid_time      datetime64[ns] 2024-01-15T12:00:00
Data variables:
    precip_rate     (latitude, longitude) float32 0.0 0.0 0.0 ... 0.0 0.0 0.0
Attributes:
    GRIB_edition:            2
    GRIB_centre:             161
    GRIB_centreDescription:  161
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             161
    history:                 2026-03-02T16:07 GRIB to CDM+CF via cfgrib-0.9.1...

# Notes
- LG paper (https://journals.ametsoc.org/view/journals/atsc/76/11/jas-d-19-0004.1.xml) uses AHPS for QPE data - National Weather Service Advanced Hydrologic Prediction Service (AHPS). 
    - But that data is daily (24h)
    - Have to do manual Z-R for LE snow vs rain (in LG paper)
    - [Gemini, need to check] AHPS: Usually a 24-hour total. It is "Gauge-Biased," meaning they take the radar and "force" it to match rain gauges.
- MRMS 
    - [Gemini] "MRMS: This is the modern successor. It provides the same radar+gauge blend but at 1-hour or even 2-minute intervals. Since your camera model is likely looking at 5- or 10-minute loops, MRMS is much better for your project than the 24-hour AHPS total mentioned in the paper.
    - Hourly! We want this version. 
- Data source: https://mesonet.agron.iastate.edu/rainfall/
    - I believe the first "About QPE estimates" is the AHPS data used by LG
    - The MRMS data blurb is below - I want to use this. 
- To consider: what QPE threshold to use to filter whether it's precipitating or not.
    - [Gemini] Is 0.1 mm/hr safe for Rain vs. Snow?The short answer is yes, but with a small caveat for snow.Rain ($0.1$ is very safe): Rain is quite "reflective." If the radar says $0.1$ mm/hr for rain, it is almost certainly raining.Snow ($0.1$ is a bit "noisy"): Snow is much less dense than rain and reflects less energy.The Risk: At very low thresholds ($< 0.1$), the radar might pick up "clear-air echoes" (dust, bugs, or even birds) that it accidentally labels as tiny amounts of snow.The Recommendation: For your "Active Storm" filter, 0.1 mm/hr is an excellent starting point for both. It is high enough to ignore most "noise" but low enough to catch "light rain" and "moderate flurries."Pro-Tip: In the weather world, we often use 0.25 mm/hr (roughly $0.01$ inches/hr) as the "standard" threshold for "measurable" precipitation. If you find $0.1$ gives you too many "Active" locations that look clear on camera, bump it up to $0.25$.